# Memvid Introductory Tutorial

## Introduction

Welcome to the Memvid Introductory Tutorial!

**What is Memvid?**
Memvid is a Python library that revolutionizes AI memory management by encoding text data into videos. The core idea is "Video-as-Database": text chunks are converted into QR codes, and these QR codes become frames in an MP4 video. This allows for:
- **Efficient Storage:** Significant compression compared to traditional databases.
- **Fast Semantic Search:** Quick retrieval of information using natural language.
- **Offline Capability:** Once built, memories can be used without internet access.
- **Portability:** Your entire knowledge base is in a few files (MP4 + index files).

**What this notebook covers:**
This notebook will walk you through the basic functionalities of Memvid:
1.  Setting up the environment in Colab.
2.  Creating a Memvid memory from simple text chunks.
3.  Searching the created memory.
4.  Interacting with the memory using the basic chat interface.
5.  Processing a sample PDF file to create another memory.
6.  Downloading your generated memory files from Colab.

Let's get started!

## Setup

First, we need to install Memvid and its dependencies. We'll also install `ffmpeg` and `libzbar0`, which are required by Memvid for video processing and QR code scanning, respectively.

In [ ]:
# Install Memvid and PyPDF2 for PDF processing
!pip install memvid PyPDF2

In [ ]:
# Install system dependencies (ffmpeg for video, zbar for QR codes)
# This command is for the Colab environment
!apt-get update && apt-get install -y ffmpeg libzbar0

Now, let's import the necessary modules from Memvid and the standard `os` module.

In [ ]:
import os
from memvid import MemvidEncoder, MemvidRetriever, MemvidChat

print("Setup Complete! Memvid modules imported.")

## 1. Creating a Memory from Text Chunks

We'll start by creating a simple Memvid memory from a list of text strings (chunks).

**The Process:**
1.  Define your text data as a list of strings.
2.  Initialize `MemvidEncoder`.
3.  Add the text chunks to the encoder using `add_chunks()`.
4.  Build the video memory and index files using `build_video()`.

In [ ]:
# Define some sample text chunks
sample_chunks = [
    "The first programmable computer was the Z3, built by Konrad Zuse.",
    "Artificial intelligence (AI) is intelligence demonstrated by machines.",
    "Memvid uses video frames to store text data encoded as QR codes.",
    "Semantic search allows finding relevant information based on meaning, not just keywords.",
    "The capital of France is Paris, known for the Eiffel Tower."
]

# Initialize the MemvidEncoder
encoder = MemvidEncoder()

# Add the chunks to the encoder
encoder.add_chunks(sample_chunks)

# Define output filenames
video_file_text = "text_memory.mp4"
# Memvid creates both .json and .faiss for the index from the base name
index_file_text_base = "text_memory_index" 

# Build the video memory
print("Building video memory from text chunks...")
encoder.build_video(video_file_text, f"{index_file_text_base}.json")
print(f"Memory built! Check for '{video_file_text}', '{index_file_text_base}.json', and '{index_file_text_base}.faiss'.")

# List files in the current directory to see the output
print("\nFiles in current directory:")
for f_name in os.listdir("."):
  if index_file_text_base in f_name or video_file_text in f_name:
    print(f"- {f_name}")

You should see `text_memory.mp4`, `text_memory_index.json`, and `text_memory_index.faiss` listed above.

## 2. Searching the Memory

Now that we have created a memory, let's search for information within it.

**The Process:**
1.  Initialize `MemvidRetriever` with the paths to your video and JSON index file (it will infer the .faiss file).
2.  Use the `search()` method with your query.

In [ ]:
# Initialize the MemvidRetriever
# Provide the .json file; it will find the .faiss file automatically
retriever_text = MemvidRetriever(video_file_text, f"{index_file_text_base}.json")

# Define a sample query
query = "What is AI?"

# Perform the search
print(f"\nSearching for: '{query}'")
results = retriever_text.search(query, top_k=2) # Get top 2 results

# Print the search results
if results:
    for i, (chunk, score) in enumerate(results):
        print(f"\nResult {i+1}:")
        print(f"  Score: {score:.4f}")
        print(f"  Chunk: {chunk}")
else:
    print("No results found.")

The retriever should find chunks related to AI from our sample data.

## 3. Chatting with the Memory (Basic)

Memvid also provides a chat interface. This allows for a more conversational way to interact with your memory, especially when integrated with a Large Language Model (LLM).

**Important Note on LLM API Keys:**
For `MemvidChat` to provide sophisticated, human-like responses, it typically needs to connect to an LLM service (like OpenAI, Google, Anthropic, etc.). This requires an API key.
- You would normally set this key as an environment variable (e.g., `os.environ['OPENAI_API_KEY'] = 'your_key_here'` or `os.environ['GOOGLE_API_KEY'] = 'your_key_here'`).
- **For this demo, we will proceed without setting an API key.**
- This means `MemvidChat` will likely use its basic mode, which might primarily return the raw context found by the retriever or a very simple response, rather than a detailed conversational reply. This still demonstrates the chat mechanism.

In [ ]:
# Initialize MemvidChat
# Since we haven't set an API key, it will use a basic fallback or no LLM.
chat_text = MemvidChat(video_file_text, f"{index_file_text_base}.json")

# Start a chat session
chat_text.start_session()
print("Chat session started (basic mode, no LLM API key).")

# Send a message to the chat
chat_query = "Tell me about Memvid."
print(f"\nUser: {chat_query}")
response = chat_text.chat(chat_query)

# Print the response
print(f"Memvid (basic response): {response}")

The response here will be basic due to the absence of an LLM, but it shows how `MemvidChat` is invoked.

## 4. Processing a PDF

Memvid can directly process PDF files, chunk their content, and build a searchable memory.

**The Process:**
1.  **Upload your PDF:** Please upload a PDF file to your Colab environment and name it `sample.pdf`. You can use the file browser panel on the left (usually a folder icon). If you don't have a PDF handy, you can skip creating the PDF-based memory for now.
2.  Initialize `MemvidEncoder`.
3.  Use `add_pdf("sample.pdf")` to process the PDF.
4.  Build the video memory.

In [ ]:
# Define the path for the user-uploaded PDF
pdf_file_path = "sample.pdf"

# Initialize a new MemvidEncoder for the PDF
encoder_pdf = MemvidEncoder()

# Check if the PDF file exists (user needs to upload it)
if os.path.exists(pdf_file_path):
    print(f"\nProcessing PDF file: '{pdf_file_path}'...")
    try:
        encoder_pdf.add_pdf(pdf_file_path)
        print(f"Content from '{pdf_file_path}' added to encoder.")
        
        # Define output filenames for the PDF memory
        video_file_pdf = "pdf_memory.mp4"
        index_file_pdf_base = "pdf_memory_index"

        # Build the video memory from the PDF content
        if encoder_pdf.get_stats()["total_chunks"] > 0:
            print("\nBuilding video memory from PDF content...")
            encoder_pdf.build_video(video_file_pdf, f"{index_file_pdf_base}.json")
            print(f"Memory built! Check for '{video_file_pdf}' and '{index_file_pdf_base}.json/.faiss'.")

            # List files in the current directory to see the new output
            print("\nFiles in current directory related to PDF memory:")
            for f_name in os.listdir("."):
              if index_file_pdf_base in f_name or video_file_pdf in f_name or pdf_file_path in f_name:
                print(f"- {f_name}")
        else:
            print("\nNo chunks were extracted from the PDF, skipping video build for PDF memory.")
            
    except Exception as e:
        print(f"Could not process {pdf_file_path}. Error: {e}")
        print("Please ensure it's a valid PDF file and all dependencies are correctly installed.")
else:
    print(f"\nPDF file '{pdf_file_path}' not found. Please upload it to your Colab session to test PDF processing.")
    # Define variables so download step doesn't fail if PDF part is skipped
    video_file_pdf = None 
    index_file_pdf_base = None

## 5. Downloading Your Memory Files

Once you've created your Memvid memories, you might want to download the MP4 video and JSON/FAISS index files to use them elsewhere. Colab provides a utility for this.

In [ ]:
from google.colab import files

print("Preparing to download memory files...")

# Files to download (check if they exist first)
files_to_download = []

# Text memory files
if os.path.exists(video_file_text): files_to_download.append(video_file_text)
if os.path.exists(f"{index_file_text_base}.json"): files_to_download.append(f"{index_file_text_base}.json")
if os.path.exists(f"{index_file_text_base}.faiss"): files_to_download.append(f"{index_file_text_base}.faiss")

# PDF memory files (if created and encoder_pdf exists)
if 'encoder_pdf' in globals() and video_file_pdf and index_file_pdf_base: 
    if os.path.exists(video_file_pdf): files_to_download.append(video_file_pdf)
    if os.path.exists(f"{index_file_pdf_base}.json"): files_to_download.append(f"{index_file_pdf_base}.json")
    if os.path.exists(f"{index_file_pdf_base}.faiss"): files_to_download.append(f"{index_file_pdf_base}.faiss")

if files_to_download:
    print(f"\nFound {len(files_to_download)} file(s) to download.")
    print("Starting downloads. Please approve each download in your browser.")
    for f_path in files_to_download:
        print(f"Downloading {f_path}...")
        files.download(f_path)
    print("\nAll specified files have been processed for download.")
else:
    print("\nNo memory files were found or created to download. Make sure the previous steps ran successfully.")

## Conclusion

Congratulations! You've completed the Memvid Introductory Tutorial.

**Recap:**
- You learned how to set up Memvid in a Colab environment.
- You created a Memvid memory from raw text chunks.
- You searched this memory using semantic queries.
- You saw how the basic chat interface works.
- You went through the process of attempting PDF ingestion (requires user upload).
- You learned how to download the generated memory files.

**Explore Further:**
This was just a basic introduction. Memvid has many more features and configuration options. Try it with your own text data or PDFs! Explore different chunking strategies, embedding models, and video settings.

For more detailed information, documentation, and advanced examples, please visit the official Memvid GitHub repository:
[https://github.com/olow304/memvid](https://github.com/olow304/memvid)

Happy memory building!